# Natural Gas Storage Contract Pricing Model
## JPMorgan Chase Quantitative Research Task

This notebook analyzes monthly natural gas prices, identifies seasonal patterns, and provides price estimates for any date (past, present, or future).

## 1. Import Required Libraries

In [1]:
import pandas as pd
import numpy as np
from scipy.interpolate import CubicSpline
from datetime import datetime, timedelta
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from statsmodels.tsa.seasonal import seasonal_decompose
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

In [ ]:
import os

# Ensure we're in the correct directory
notebook_dir = os.path.dirname(os.path.abspath('Nat_Gas.csv'))
if not os.path.exists('Nat_Gas.csv'):
    # Try the current working directory
    script_dir = os.path.dirname(os.path.abspath(__file__))
    csv_path = os.path.join(script_dir, 'Nat_Gas.csv')
else:
    csv_path = 'Nat_Gas.csv'

print(f"Using CSV file: {csv_path}")
print(f"File exists: {os.path.exists(csv_path)}")

## 2. Define the NaturalGasPricingModel Class

In [2]:
class NaturalGasPricingModel:
    def __init__(self, csv_path):
        self.df = pd.read_csv(csv_path)
        self._preprocess_data()
        self._analyze_seasonality()
        self._build_extrapolation_model()
    
    def _preprocess_data(self):
        self.df['Date'] = pd.to_datetime(self.df['Dates'], format='%m/%d/%y')
        self.df['Price'] = pd.to_numeric(self.df['Prices'], errors='coerce')
        self.df = self.df.sort_values('Date').reset_index(drop=True)
        self.df['Year'] = self.df['Date'].dt.year
        self.df['Month'] = self.df['Date'].dt.month
        self.df['DayOfYear'] = self.df['Date'].dt.dayofyear
        self.min_date = self.df['Date'].min()
        self.max_date = self.df['Date'].max()
        self.extrapolation_end = self.max_date + timedelta(days=365)
        print(f'Data loaded: {self.min_date.date()} to {self.max_date.date()}')
        print(f'Price range: ${self.df["Price"].min():.2f} - ${self.df["Price"].max():.2f}')
        print(f'Number of data points: {len(self.df)}')

In [3]:
    def _analyze_seasonality(self):
        self.seasonal_avg = self.df.groupby('Month')['Price'].mean()
        ts_data = self.df.set_index('Date')['Price']
        try:
            decomposition = seasonal_decompose(ts_data, model='additive', period=12, extrapolate='extend')
        except TypeError:
            decomposition = seasonal_decompose(ts_data, model='additive', period=12)
        self.trend = decomposition.trend
        self.seasonal = decomposition.seasonal
        self.residual = decomposition.resid
        print('\n--- Seasonal Analysis ---')
        print('Average Price by Month:')
        for month in range(1, 13):
            month_name = pd.Timestamp(2024, month, 1).strftime('%B')
            avg_price = self.seasonal_avg.get(month, np.nan)
            if not np.isnan(avg_price):
                print(f'  {month_name:12s}: ${avg_price:7.2f}')

In [4]:
    def _build_extrapolation_model(self):
        self.dates_numeric = np.array([(d - self.min_date).days for d in self.df['Date']])
        self.spline = CubicSpline(self.dates_numeric, self.df['Price'].values, bc_type='natural')
        last_6_months = self.df.tail(6)
        x_vals = self.dates_numeric[-6:]
        y_vals = last_6_months['Price'].values
        coeffs = np.polyfit(x_vals, y_vals, 1)
        self.trend_slope = coeffs[0]
        self.trend_intercept = coeffs[1]

In [5]:
    def get_price(self, date):
        if isinstance(date, str):
            date = pd.to_datetime(date)
        days_from_start = (date - self.min_date).days
        if self.min_date <= date <= self.max_date:
            return float(self.spline(days_from_start))
        elif date <= self.extrapolation_end:
            return self._extrapolate_price(date, days_from_start)
        else:
            raise ValueError(f'Date {date.date()} is beyond extrapolation range')

In [6]:
    def _extrapolate_price(self, date, days_from_start):
        last_day_numeric = self.dates_numeric[-1]
        days_beyond = days_from_start - last_day_numeric
        trend_component = self.df['Price'].iloc[-1] + self.trend_slope * days_beyond
        month = date.month
        seasonal_adjustment = self.seasonal_avg.get(month, 0) - self.seasonal_avg.mean()
        extrapolated_price = trend_component + (seasonal_adjustment * 0.3)
        extrapolated_price = max(extrapolated_price, 0.5)
        return float(extrapolated_price)

In [7]:
    def get_price_range(self, start_date, end_date, frequency='D'):
        date_range = pd.date_range(start=start_date, end=end_date, freq=frequency)
        prices = [self.get_price(d) for d in date_range]
        return pd.DataFrame({'Date': date_range, 'Price': prices})

In [8]:
    def storage_contract_analysis(self, injection_date, withdrawal_date):
        injection_price = self.get_price(injection_date)
        withdrawal_price = self.get_price(withdrawal_date)
        days_held = (pd.to_datetime(withdrawal_date) - pd.to_datetime(injection_date)).days
        months_held = days_held / 30
        storage_cost = months_held * 0.50
        price_spread = withdrawal_price - injection_price
        net_spread = price_spread - storage_cost
        return {
            'injection_date': injection_date,
            'injection_price': round(injection_price, 2),
            'withdrawal_date': withdrawal_date,
            'withdrawal_price': round(withdrawal_price, 2),
            'days_held': days_held,
            'price_spread': round(price_spread, 2),
            'estimated_storage_cost': round(storage_cost, 2),
            'net_profit_potential': round(net_spread, 2),
            'profitable': net_spread > 0
        }

## 3. Load Data and Initialize Model

In [ ]:
print('=' * 70)
print('NATURAL GAS STORAGE CONTRACT PRICING MODEL')
print('=' * 70)
model = NaturalGasPricingModel(csv_path)

NATURAL GAS STORAGE CONTRACT PRICING MODEL
Data loaded: 2020-10-31 to 2024-09-30
Price range: $9.84 - $12.80
Number of data points: 48


AttributeError: 'NaturalGasPricingModel' object has no attribute '_analyze_seasonality'

## 4. Price Estimation Examples

In [ ]:
print('\n' + '=' * 70)
print('PRICE ESTIMATION EXAMPLES')
print('=' * 70)
test_dates = ['2020-10-31', '2021-06-15', '2024-09-30', '2024-12-15', '2025-06-30', '2025-09-30']
print('\nSample Price Estimates:')
for test_date in test_dates:
    price = model.get_price(test_date)
    date_obj = pd.to_datetime(test_date)
    if date_obj <= model.max_date:
        data_type = '(Historical)'
    else:
        data_type = '(Extrapolated)'
    print(f'  {test_date} {data_type}: ${price:7.2f}/MMBtu')

## 5. Storage Contract Analysis

In [ ]:
print('\n' + '=' * 70)
print('STORAGE CONTRACT EXAMPLE')
print('=' * 70)
contract_analysis = model.storage_contract_analysis('2024-06-30', '2024-12-31')
print('\nSummer-to-Winter Storage Strategy:')
for key, value in contract_analysis.items():
    if key != 'profitable':
        print(f'  {key.replace("_", " ").title():25s}: {value}')
    else:
        status = 'Yes' if value else 'No'
        print(f'  Profitable: {status}')

## 6. Generate Extended Price Dataset

In [ ]:
print('\n' + '=' * 70)
print('GENERATING EXTENDED PRICE DATASET')
print('=' * 70)
extended_prices = model.get_price_range('2020-10-31', '2025-09-30', frequency='MS')
extended_prices.to_csv('extended_nat_gas_prices.csv', index=False)
print(f'Extended price data saved ({len(extended_prices)} data points)')

In [ ]:
print('\nFirst few extended prices:')
display(extended_prices.head(10))

In [ ]:
print('\nLast few extended prices:')
display(extended_prices.tail(10))

## 7. Analysis Complete

In [ ]:
print('\n' + '=' * 70)
print('ANALYSIS COMPLETE')
print('=' * 70)
print('\nKey Findings:')
print(f'- Data spans {len(model.df)} months from {model.min_date.date()} to {model.max_date.date()}')
print(f'- Historical price range: ${model.df["Price"].min():.2f} - ${model.df["Price"].max():.2f}/MMBtu')
print(f'- Winter months (Dec-Feb) show higher average prices')
print(f'- Summer months (Jun-Aug) show lower average prices')
print(f'- Model provides 1-year extrapolation through {model.extrapolation_end.date()}')